In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!pip install catboost

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report

# 1. 데이터 불러오기
train_df = pd.read_csv('/content/drive/MyDrive/블루문/card_train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/블루문/card_test_remove.csv')

# 2. '이용금액' 컬럼 숫자형 변환
amount_cols = [c for c in train_df.columns if '이용금액' in c]
for col in amount_cols:
    train_df[col] = pd.to_numeric(train_df[col], errors='coerce')
    if col in test_df.columns:
        test_df[col] = pd.to_numeric(test_df[col], errors='coerce')

# 3. 시계열(기간별) 파생 피처 생성 (R12M, R6M, R3M, B0M 등)
def add_time_features(df):
    df = df.copy()
    for base in ['일시불', '할부', 'CA', '체크', '카드론']:
        try:
            r12 = f'이용금액_{base}_R12M'
            r6 = f'이용금액_{base}_R6M'
            r3 = f'이용금액_{base}_R3M'
            b0 = f'이용금액_{base}_B0M'
            if all(x in df.columns for x in [r12, r6, r3, b0]):
                df[f'{base}_3M12M_비율'] = df[r3] / (df[r12] + 1e-5)
                df[f'{base}_3M6M_비율'] = df[r3] / (df[r6] + 1e-5)
                df[f'{base}_B0M3M_비율'] = df[b0] / (df[r3] + 1e-5)
                df[f'{base}_3M_증감'] = df[r3] - df[r6]
                df[f'{base}_12M_월평균'] = df[r12] / 12
        except Exception:
            pass
    return df

train_df = add_time_features(train_df)
test_df = add_time_features(test_df)

# 4. 기존 파생 피처 생성
def add_features(df):
    df = df.copy()
    homecare_cols = [c for c in ['납부_관리비이용금액', '납부_가스전기료이용금액', '납부_보험료이용금액'] if c in df.columns]
    df['홈케어_합계'] = df[homecare_cols].sum(axis=1)
    for cat in ['쇼핑', '교통', '여유', '납부']:
        cols = [c for c in amount_cols if c.startswith(f'{cat}_') and c in df.columns]
        df[f'{cat}_합계'] = df[cols].sum(axis=1)
    df['총_이용금액'] = df[[c for c in amount_cols if c in df.columns]].sum(axis=1)
    for cat in ['쇼핑', '교통', '여유', '납부', '홈케어']:
        col_sum = f'{cat}_합계'
        if col_sum in df.columns:
            df[f'{cat}_비율'] = df[col_sum] / (df['총_이용금액'] + 1e-5)
            df[f'{cat}_비율_log'] = np.log1p(df[f'{cat}_비율'])
            df[f'{cat}_비율_sq'] = df[f'{cat}_비율'] ** 2
    if '연령' in df.columns:
        df['연령'] = pd.to_numeric(df['연령'], errors='coerce')
        df['연령대'] = (df['연령'].fillna(0) // 10) * 10
    df['고액결제비율'] = (df[amount_cols] > 100000).sum(axis=1) / len(amount_cols)
    if '쇼핑_비율' in df.columns and '연령대' in df.columns:
        df['연령x쇼핑'] = df['연령대'] * df['쇼핑_비율']
    if '여유_비율' in df.columns and '연령대' in df.columns:
        df['연령x여유'] = df['연령대'] * df['여유_비율']
    if '쇼핑_비율' in df.columns and '교통_비율' in df.columns:
        df['쇼핑/교통'] = df['쇼핑_비율'] / (df['교통_비율'] + 1e-5)
    return df

train_df = add_features(train_df)
test_df = add_features(test_df)

# 5. 피처 리스트(기간별 파생 포함)
base_cols = ['연령', '남녀구분코드', '거주시도명', '직장시도명', 'Life_Stage']
add_cols = [
    '쇼핑_합계', '교통_합계', '여유_합계', '납부_합계', '홈케어_합계', '총_이용금액',
    '쇼핑_비율', '교통_비율', '여유_비율', '납부_비율', '홈케어_비율', '연령대', '고액결제비율',
    '연령x쇼핑', '연령x여유', '쇼핑/교통'
]
# 시계열 파생 피처 자동 포함
time_cols = [c for c in train_df.columns if any(x in c for x in ['3M12M_비율', '3M6M_비율', 'B0M3M_비율', '3M_증감', '12M_월평균'])]
lifestyle_cols = [col for col in (base_cols + add_cols + time_cols) if col in train_df.columns and col in test_df.columns]

# 6. 결측치 처리(중앙값), 인코딩, 스케일링
for col in lifestyle_cols:
    if col in train_df.columns:
        if train_df[col].dtype == 'O':
            train_df[col] = train_df[col].fillna('미상')
            test_df[col] = test_df[col].fillna('미상')
        else:
            median = train_df[col].median()
            train_df[col] = train_df[col].fillna(median)
            test_df[col] = test_df[col].fillna(median)

cat_cols = [col for col in ['남녀구분코드', '거주시도명', '직장시도명', 'Life_Stage', '연령대'] if col in lifestyle_cols]
for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))

num_cols = [col for col in lifestyle_cols if col not in cat_cols and col != '연령']
for col in num_cols:
    train_df[col] = train_df[col].clip(lower=0)
    test_df[col] = test_df[col].clip(lower=0)
train_df[num_cols] = np.log1p(train_df[num_cols])
test_df[num_cols] = np.log1p(test_df[num_cols])
for col in num_cols:
    median = train_df[col].median()
    train_df[col] = train_df[col].replace([np.inf, -np.inf], np.nan).fillna(median)
    test_df[col] = test_df[col].replace([np.inf, -np.inf], np.nan).fillna(median)
scaler = StandardScaler()
train_df[num_cols + ['연령']] = scaler.fit_transform(train_df[num_cols + ['연령']])
test_df[num_cols + ['연령']] = scaler.transform(test_df[num_cols + ['연령']])

target_col = 'Segment' if 'Segment' in train_df.columns else 'Segment.1'
X = train_df[lifestyle_cols]
y = train_df[target_col]

# 7. 교차검증 + LGBM/RandomForest/CatBoost 앙상블
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lgb_preds, rf_preds, cat_preds = [], [], []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgb_model = lgb.LGBMClassifier(
        objective='multiclass',
        num_class=5,
        class_weight={'A':10, 'B':5, 'C':1, 'D':1, 'E':1},
        random_state=fold,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=8,
        num_leaves=40,
        min_child_samples=20
    )
    lgb_model.fit(X_train, y_train, categorical_feature=cat_cols)
    lgb_pred = lgb_model.predict_proba(X_val)
    lgb_preds.append(lgb_pred)

    rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=fold)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict_proba(X_val)
    rf_preds.append(rf_pred)

    cat_model = CatBoostClassifier(
        iterations=200, depth=8, learning_rate=0.05, random_seed=fold,
        verbose=0, auto_class_weights='Balanced'
    )
    cat_model.fit(X_train, y_train)
    cat_pred = cat_model.predict_proba(X_val)
    cat_preds.append(cat_pred)

    # fold별 성능 출력
    y_pred = np.argmax((lgb_pred + rf_pred + cat_pred) / 3, axis=1)
    labels = lgb_model.classes_
    y_pred = labels[y_pred]
    print(f"[Fold {fold+1}]")
    print(classification_report(y_val, y_pred))

# 8. 전체 학습 후 테스트 예측 (앙상블)
lgb_model.fit(X, y, categorical_feature=cat_cols)
rf_model.fit(X, y)
cat_model.fit(X, y)
lgb_test = lgb_model.predict_proba(test_df[lifestyle_cols])
rf_test = rf_model.predict_proba(test_df[lifestyle_cols])
cat_test = cat_model.predict_proba(test_df[lifestyle_cols])
ensemble_test_pred = np.argmax((lgb_test + rf_test + cat_test) / 3, axis=1)
labels = lgb_model.classes_
ensemble_test_pred = labels[ensemble_test_pred]

df_submit = pd.DataFrame({'ID': test_df['ID'], 'Segment': ensemble_test_pred})
df_submit.to_csv('submission.csv', index=False)

/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result 

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.072385 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8535
[LightGBM] [Info] Number of data points in the train set: 56448, number of used features: 41
[LightGBM] [Info] Start training from score -5.551232
[LightGBM] [Info] Start training from score -7.949127
[LightGBM] [Info] Start training from score -2.937825
[LightGBM] [Info] Start training from score -1.931020
[LightGBM] [Info] Start training from score -0.225918
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018561 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8537
[LightGBM] [Info] Number of data points in the train set: 56448, number of used features: 41
[LightGBM] [Info] Start training from score -5.506868
[LightGBM] [Info] Start training from score -8.236897
[LightGBM] [Info] Start training from score -2.937913
[LightGBM] [Info] Start training from score -1.931109
[LightGBM] [Info] Start training from score -0.226006
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018112 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8523
[LightGBM] [Info] Number of data points in the train set: 56448, number of used features: 41
[LightGBM] [Info] Start training from score -5.506868
[LightGBM] [Info] Start training from score -8.236897
[LightGBM] [Info] Start training from score -2.937913
[LightGBM] [Info] Start training from score -1.931109
[LightGBM] [Info] Start training from score -0.226006
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017934 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8533
[LightGBM] [Info] Number of data points in the train set: 56448, number of used features: 41
[LightGBM] [Info] Start training from score -5.551161
[LightGBM] [Info] Start training from score -8.236738
[LightGBM] [Info] Start training from score -2.937421
[LightGBM] [Info] Start training from score -1.930950
[LightGBM] [Info] Start training from score -0.225848
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022478 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8599
[LightGBM] [Info] Number of data points in the train set: 70560, number of used features: 41
[LightGBM] [Info] Start training from score -5.533220
[LightGBM] [Info] Start training from score -8.172277
[LightGBM] [Info] Start training from score -2.937699
[LightGBM] [Info] Start training from score -1.931027
[LightGBM] [Info] Start training from score -0.225925
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

In [8]:
# 예측 결과 비율 확인
print(df_submit['Segment'].value_counts().sort_index())
print(df_submit['Segment'].value_counts(normalize=True).sort_index())

Segment
C      65
D     217
E    1158
Name: count, dtype: int64
Segment
C    0.045139
D    0.150694
E    0.804167
Name: proportion, dtype: float64


In [9]:
import pandas as pd

# 1. 정답지와 예측 결과 불러오기
answer = pd.read_csv('/content/drive/MyDrive/블루문/answer_key.csv')
pred = pd.read_csv('/content/submission.csv')

# 2. ID 컬럼을 문자열로 통일
answer['ID'] = answer['ID'].astype(str)
pred['ID'] = pred['ID'].astype(str)

# 3. ID 기준 병합
compare = answer.merge(pred, on='ID', how='left')

# 정답 여부 컬럼 생성
compare['correct'] = compare['Segment_x'] == compare['Segment_y']

# 정답률(정확도) 계산 및 출력
accuracy = compare['correct'].mean()
num_correct = compare['correct'].sum()
total = len(compare)
print(f'정답과 예측 일치 비율: {accuracy:.4f} ({num_correct} / {total})')

정답과 예측 일치 비율: 0.8608 (1243 / 1444)
